In [2]:
# Install packages
from pathlib import Path
import pandas as pd
import geopandas as gpd
import seaborn as sns

#Set file paths
root_fldr = Path.cwd().parent
raw_fldr = root_fldr/"data"/"raw"
processed_fldr = root_fldr/'data'/'processed'


rxburn_gdb = raw_fldr/'fire25_1.gdb'
prescribed_burns = gpd.read_file(rxburn_gdb, layer = "rxburn25_1")
prescribed_burns.head()

c:\Users\jmg177\AppData\Local\miniforge3\envs\env700\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


,YEAR_,STATE,AGENCY,UNIT_ID,TREATMENT_ID,TREATMENT_NAME,TREATMENT_TYPE,START_DATE,END_DATE,TREATED_AC,GIS_ACRES,RX_CONSUM,PRE_CON_CLASS,POST_CON_CLASS,Shape_Length,Shape_Area,geometry
0,2025.0,CA,BLM,CND,38441,BIL312D Stornetta Rx_FY25,3.0,2024-10-01 00:00:00+00:00,2025-02-01 00:00:00+00:00,15.0,14.621698,3.0,NaN,NaN,1104.323979,59171.913798,"MULTIPOLYGON (((-321330.161 106540.219, -32131..."
1,2025.0,CA,BLM,NOD,45361,BIL312D_FY25_Butte Pile RX,3.0,2024-10-01 00:00:00+00:00,2025-02-01 00:00:00+00:00,50.0,49.999985,4.0,NaN,NaN,1799.301659,202342.766575,"MULTIPOLYGON (((-115147.027 171857.101, -11469..."
2,2025.0,CA,BLM,NOD,38421,BIL312D_FY25_North Horse Pile RX,3.0,2025-02-10 00:00:00+00:00,2025-02-12 00:00:00+00:00,229.0,228.603134,4.0,NaN,NaN,5386.383188,925124.048407,"MULTIPOLYGON (((-34815.102 297254.474, -34654...."
3,2025.0,CA,BLM,NOD,40041,BIL312D_AGFO_WUI_Machine_Pile_Burn,3.0,2024-10-01 00:00:00+00:00,2025-02-13 00:00:00+00:00,116.0,116.124580,4.0,NaN,NaN,3960.591710,469939.500485,"MULTIPOLYGON (((-86638.188 344095.855, -86636...."
4,2025.0,CA,BLM,CND,41684,Black Forest Rx_FY25,3.0,2024-10-01 00:00:00+00:00,2025-03-25 00:00:00+00:00,25.0,25.118351,4.0,NaN,NaN,3452.593270,101650.358387,"MULTIPOLYGON (((-238491.591 112458.684, -23848..."


In [3]:
prescribed_burns.columns

Index(['YEAR_', 'STATE', 'AGENCY', 'UNIT_ID', 'TREATMENT_ID', 'TREATMENT_NAME',
       'TREATMENT_TYPE', 'START_DATE', 'END_DATE', 'TREATED_AC', 'GIS_ACRES',
       'RX_CONSUM', 'PRE_CON_CLASS', 'POST_CON_CLASS', 'Shape_Length',
       'Shape_Area', 'geometry'],
      dtype='str')

In [4]:
# Check missing values
prescribed_burns.isna().mean().sort_values()*100

STATE              0.000000
AGENCY             0.000000
Shape_Area         0.000000
Shape_Length       0.000000
geometry           0.000000
GIS_ACRES          0.016701
UNIT_ID            0.150313
YEAR_              0.175365
TREATMENT_ID       3.440501
TREATED_AC         6.555324
TREATMENT_NAME    12.258873
START_DATE        14.805846
END_DATE          16.475992
TREATMENT_TYPE    17.661795
RX_CONSUM         90.972860
POST_CON_CLASS    95.791232
PRE_CON_CLASS     95.791232
dtype: float64

Columns with over 90% missing values will not be useful for analysis, so those columns will not be analyzed. Year is necessary for analysis, so the few rows with no year will  be dropped.

In [6]:
#Cleaning
prescribed_burns_cleaned = (
    prescribed_burns
    .dropna(subset=["YEAR_"])
    .drop(columns = ["STATE", "UNIT_ID", "TREATMENT_ID", "RX_CONSUM", "PRE_CON_CLASS", "POST_CON_CLASS"])
    .rename(columns = {
            "YEAR_" : "year",
            "AGENCY" : "agency",
            "TREATMENT_NAME" : "burn_name",
            "TREATMENT_TYPE" : "treatment_type",
            "START_DATE" : "start_date",
            "END_DATE" : "end_date",
            "TREATED_AC" : "acres_treated",
            "GIS_ACRES" : "gis_acres_treated"
            })
    .assign(
        year=lambda df: df["year"].astype(int)
    )
)

In [7]:
prescribed_burns_cleaned.head()

,year,agency,burn_name,treatment_type,start_date,end_date,acres_treated,gis_acres_treated,Shape_Length,Shape_Area,geometry
0,2025,BLM,BIL312D Stornetta Rx_FY25,3.0,2024-10-01 00:00:00+00:00,2025-02-01 00:00:00+00:00,15.0,14.621698,1104.323979,59171.913798,"MULTIPOLYGON (((-321330.161 106540.219, -32131..."
1,2025,BLM,BIL312D_FY25_Butte Pile RX,3.0,2024-10-01 00:00:00+00:00,2025-02-01 00:00:00+00:00,50.0,49.999985,1799.301659,202342.766575,"MULTIPOLYGON (((-115147.027 171857.101, -11469..."
2,2025,BLM,BIL312D_FY25_North Horse Pile RX,3.0,2025-02-10 00:00:00+00:00,2025-02-12 00:00:00+00:00,229.0,228.603134,5386.383188,925124.048407,"MULTIPOLYGON (((-34815.102 297254.474, -34654...."
3,2025,BLM,BIL312D_AGFO_WUI_Machine_Pile_Burn,3.0,2024-10-01 00:00:00+00:00,2025-02-13 00:00:00+00:00,116.0,116.124580,3960.591710,469939.500485,"MULTIPOLYGON (((-86638.188 344095.855, -86636...."
4,2025,BLM,Black Forest Rx_FY25,3.0,2024-10-01 00:00:00+00:00,2025-03-25 00:00:00+00:00,25.0,25.118351,3452.593270,101650.358387,"MULTIPOLYGON (((-238491.591 112458.684, -23848..."
